In [5]:
import pandas as pd


In [6]:
df = pd.read_csv('../Data/olist_orders_dataset.csv')
df.info()
display(df.head())


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [11]:
missing_value = df.isnull().sum()
duplicate_orders = df['order_id'].duplicated().sum()


In [14]:
time_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']

for col in time_cols:
    df[col] = pd.to_datetime(df[col])

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


## Data Contract: `orders` Domain

**Topic Name:** `orders.v1`
**Source:** Olist E-commerce Dataset (`olist_orders_dataset.csv`)

### Overview
This contract defines the strict schema constraints required for producing events to the `orders.v1` Kafka topic. It is based on upstream profiling of 99,441 raw records.

### Schema Definition
| Field Name | Source Type (Pandas) | Target Type (Avro) | Nullable? | Business Context / Rules |
| :--- | :--- | :--- | :--- | :--- |
| `order_id` | `str` | `string` | **No** | **Primary Key.** Must be unique. 0 missing values detected. |
| `customer_id` | `str` | `string` | **No** | **Foreign Key.** Links to the customer domain. |
| `order_status` | `str` | `string` | **No** | Categorical state (e.g., 'delivered', 'shipped', 'canceled'). |
| `order_purchase_timestamp` | `datetime64[us]` | `string` | **No** | ISO 8601 formatted string representing the exact checkout time. |
| `order_estimated_delivery_date` | `datetime64[us]` | `string` | **No** | ISO 8601 formatted string. Always present at checkout. |
| `order_approved_at` | `datetime64[us]` | `["null", "string"]` | **Yes** | Null if payment failed or is still pending. (160 nulls detected). |
| `order_delivered_carrier_date` | `datetime64[us]` | `["null", "string"]` | **Yes** | Null if order was canceled before shipping. (1,783 nulls detected). |
| `order_delivered_customer_date` | `datetime64[us]` | `["null", "string"]` | **Yes** | Null if order is in-transit, lost, or canceled. (2,965 nulls detected). |


### Engineering Notes
* **Timestamps:** While Pandas profiled the timestamps as `datetime64[us]`, Avro natively lacks a robust logical type for dates across all languages. Therefore, we cast all datetimes back to ISO 8601 `string` types in the producer payload for maximum cross-language compatibility downstream.
* **Null Handling:** Missing dates must be serialized as a native Python `None` (not `NaN` or `NaT`) to satisfy the Avro union type `["null", "string"]`.t
